# Complete Profit Analysis - All Countries

Comprehensive profit analysis combining:
- Marketplace sales data (GMU + df_10000) from all countries
- Provider costs data
- Currency conversion for PL and CZ
- Profit calculations for PAID and UNPAID orders

In [1]:
import pandas as pd
import numpy as np
import glob
from datetime import timedelta

# Exchange rates (EUR conversion)
EXCHANGE_RATES = {
    'PL': 4.2,   # EUR to PLN
    'CZ': 24.3,  # EUR to CZK
    'DE': 1.0,
    'FR': 1.0,
    'IT': 1.0,
    'AT': 1.0,
    'SK': 1.0
}

print("Exchange Rates (to EUR):")
for country, rate in EXCHANGE_RATES.items():
    print(f"  {country}: {rate}")

Exchange Rates (to EUR):
  PL: 4.2
  CZ: 24.3
  DE: 1.0
  FR: 1.0
  IT: 1.0
  AT: 1.0
  SK: 1.0


In [2]:
# Multi-language keyword patterns
RELEASE_KEYWORDS = ['Freigabe', 'Release', 'Rilascio', 'Libération', 'Zwolnienie', 'Uvolnění', 'Uvoľnenie']
GOODS_RECEIPT_KEYWORDS = ['Wareneingang', 'Goods receipt', 'Entrata merce', 'Réception', 'Przyjęcie', 'Příjem', 'Príjem']
COMMISSION_KEYWORDS = ['Provision', 'commission', 'Commissione', 'Commission', 'Prowizja', 'Provize']

def contains_any(text, keywords):
    """Check if text contains any of the keywords (case insensitive)"""
    if pd.isna(text):
        return False
    text_lower = str(text).lower()
    return any(keyword.lower() in text_lower for keyword in keywords)

print("✓ Utility functions loaded")

✓ Utility functions loaded


In [3]:
# Load ALL marketplace data from all countries
print("="*70)
print("LOADING MARKETPLACE DATA FROM ALL COUNTRIES")
print("="*70)

countries = ['de', 'fr', 'it', 'at', 'cz', 'pl', 'sk']
all_gmu_data = []
all_df10000_data = []

for country in countries:
    gmu_files = glob.glob(f'report_booking_gmu_{country}_*.csv')
    df10k_files = glob.glob(f'report_booking10000_{country}_*.csv')
    
    if len(gmu_files) > 0 and len(df10k_files) > 0:
        # Load GMU
        df_gmu = pd.read_csv(gmu_files[0], sep=';', decimal=',', thousands='.', encoding='utf-8')
        df_gmu['country'] = country.upper()
        df_gmu['booking_date'] = pd.to_datetime(df_gmu['booking_date'])
        df_gmu['order_date'] = pd.to_datetime(df_gmu['order_date'])
        all_gmu_data.append(df_gmu)
        
        # Load df_10000
        df_10k = pd.read_csv(df10k_files[0], sep=';', decimal=',', thousands='.', encoding='utf-8')
        df_10k['country'] = country.upper()
        df_10k['Datum'] = pd.to_datetime(df_10k['Datum'])
        all_df10000_data.append(df_10k)
        
        print(f"  {country.upper()}: GMU={len(df_gmu)} rows, df_10000={len(df_10k)} rows")

# Combine all data
df_gmu_all = pd.concat(all_gmu_data, ignore_index=True)
df_10000_all = pd.concat(all_df10000_data, ignore_index=True)

print(f"\n✓ Total GMU rows: {len(df_gmu_all)}")
print(f"✓ Total df_10000 rows: {len(df_10000_all)}")

LOADING MARKETPLACE DATA FROM ALL COUNTRIES
  DE: GMU=44 rows, df_10000=101 rows
  FR: GMU=10 rows, df_10000=29 rows
  IT: GMU=2 rows, df_10000=21 rows
  AT: GMU=42 rows, df_10000=98 rows
  CZ: GMU=35 rows, df_10000=117 rows
  PL: GMU=55 rows, df_10000=184 rows
  SK: GMU=50 rows, df_10000=141 rows

✓ Total GMU rows: 238
✓ Total df_10000 rows: 691


In [4]:
# Identify PAID vs UNPAID vs CANCELLED orders across all countries
print("="*70)
print("IDENTIFYING ORDER STATUS (ALL COUNTRIES)")
print("="*70)

# Get order sets
gmu_orders = set(df_gmu_all['order_number'].dropna().unique())

# PAID: Orders with Freigabe/Release (Sales Released) transaction
freigabe_orders = set(
    df_10000_all[df_10000_all['Buchungstext'].apply(lambda x: contains_any(x, RELEASE_KEYWORDS))]['Bestellnummer'].dropna().unique()
)

# Created: Orders with Wareneingang/Goods receipt
wareneingang_orders = set(
    df_10000_all[df_10000_all['Buchungstext'].apply(lambda x: contains_any(x, GOODS_RECEIPT_KEYWORDS))]['Bestellnummer'].dropna().unique()
)

# CANCELLED: Orders with "Cancel" in Buchungstext
CANCEL_KEYWORDS = ['Cancel', 'Storno', 'Annul', 'Anulacja', 'Annulation']
cancelled_orders = set(
    df_10000_all[df_10000_all['Buchungstext'].apply(lambda x: contains_any(x, CANCEL_KEYWORDS))]['Bestellnummer'].dropna().unique()
)

print(f"\nTotal unique orders in GMU: {len(gmu_orders)}")
print(f"Orders with Wareneingang (created): {len(wareneingang_orders)}")
print(f"Orders with Freigabe/Release (PAID): {len(freigabe_orders)}")
print(f"Orders with Cancel (CANCELLED): {len(cancelled_orders)}")

# UNPAID = Created but not paid AND not cancelled
unpaid_orders_set = (wareneingang_orders - freigabe_orders) - cancelled_orders
print(f"\nUNPAID orders (created, not paid, not cancelled): {len(unpaid_orders_set)}")

IDENTIFYING ORDER STATUS (ALL COUNTRIES)

Total unique orders in GMU: 137
Orders with Wareneingang (created): 221
Orders with Freigabe/Release (PAID): 137
Orders with Cancel (CANCELLED): 56

UNPAID orders (created, not paid, not cancelled): 42


In [5]:
# Build complete order list with proper status classification
print("="*70)
print("BUILDING COMPLETE ORDER LIST")
print("="*70)

# Start with GMU sales
gmu_sales = df_gmu_all[df_gmu_all['order_number'].notna()].copy()

# Classify order status
gmu_sales['is_paid'] = gmu_sales['order_number'].isin(freigabe_orders)
gmu_sales['is_cancelled'] = gmu_sales['order_number'].isin(cancelled_orders)
gmu_sales['order_status'] = 'UNKNOWN'
gmu_sales.loc[gmu_sales['is_paid'], 'order_status'] = 'PAID'
gmu_sales.loc[gmu_sales['is_cancelled'], 'order_status'] = 'CANCELLED'
gmu_sales.loc[(~gmu_sales['is_paid']) & (~gmu_sales['is_cancelled']), 'order_status'] = 'UNPAID'

gmu_sales['source'] = 'GMU'

print(f"\nOrders from GMU: {len(gmu_sales)}")
print(f"  By country:")
print(gmu_sales.groupby('country')['order_number'].count())
print(f"\n  By status:")
print(gmu_sales['order_status'].value_counts())

# Find orders only in df_10000 (recent orders not yet in GMU)
orders_only_in_df10000 = wareneingang_orders - gmu_orders
print(f"\nOrders ONLY in df_10000 (not yet in GMU): {len(orders_only_in_df10000)}")

if len(orders_only_in_df10000) > 0:
    wareneingang_df = df_10000_all[df_10000_all['Buchungstext'].apply(lambda x: contains_any(x, GOODS_RECEIPT_KEYWORDS))].copy()
    commission_df = df_10000_all[df_10000_all['Buchungstext'].apply(lambda x: contains_any(x, COMMISSION_KEYWORDS))].copy()
    
    recent_orders_list = []
    for order_num in orders_only_in_df10000:
        waren_row = wareneingang_df[wareneingang_df['Bestellnummer'] == order_num]
        comm_row = commission_df[commission_df['Bestellnummer'] == order_num]
        
        if len(waren_row) > 0:
            row = waren_row.iloc[0]
            commission = abs(comm_row.iloc[0]['Betrag']) if len(comm_row) > 0 else 0
            gross = row['Betrag']
            
            # Determine status
            is_paid = order_num in freigabe_orders
            is_cancelled = order_num in cancelled_orders
            if is_paid:
                status = 'PAID'
            elif is_cancelled:
                status = 'CANCELLED'
            else:
                status = 'UNPAID'
            
            recent_orders_list.append({
                'order_number': order_num,
                'booking_date': row['Datum'],
                'order_date': row['Datum'],
                'booking_text': row['Buchungstext'],
                'price_gross': gross,
                'sum_price_gross': gross,
                'fee_gross': commission,
                'payout': gross - commission,
                'is_paid': is_paid,
                'is_cancelled': is_cancelled,
                'order_status': status,
                'source': 'df_10000_only',
                'country': row['country'],
                'title_item': 'See df_10000 for details',
                'shipping_charges_gross': 0,
                'shipping.first_name': '',
                'shipping.last_name': '',
                'shipping.city': ''
            })
    
    recent_orders_df = pd.DataFrame(recent_orders_list)
    all_orders = pd.concat([gmu_sales, recent_orders_df], ignore_index=True)
    print(f"  Created {len(recent_orders_df)} entries from df_10000")
    print(f"    Status breakdown:")
    print(recent_orders_df['order_status'].value_counts())
else:
    all_orders = gmu_sales

# Add month column
all_orders['month'] = all_orders['booking_date'].dt.to_period('M')

print(f"\n{'='*50}")
print(f"FINAL ORDER CLASSIFICATION")
print(f"{'='*50}")
print(f"Total orders: {len(all_orders)}")
print(f"\nBy status:")
print(all_orders['order_status'].value_counts())
print(f"\nPAID: {len(all_orders[all_orders['order_status'] == 'PAID'])}")
print(f"UNPAID: {len(all_orders[all_orders['order_status'] == 'UNPAID'])}")
print(f"CANCELLED: {len(all_orders[all_orders['order_status'] == 'CANCELLED'])}")

BUILDING COMPLETE ORDER LIST

Orders from GMU: 137
  By country:
country
AT    24
CZ    19
DE    29
FR     1
IT     1
PL    32
SK    31
Name: order_number, dtype: int64

  By status:
order_status
PAID    137
Name: count, dtype: int64

Orders ONLY in df_10000 (not yet in GMU): 98
  Created 98 entries from df_10000
    Status breakdown:
order_status
CANCELLED    56
UNPAID       42
Name: count, dtype: int64

FINAL ORDER CLASSIFICATION
Total orders: 235

By status:
order_status
PAID         137
CANCELLED     56
UNPAID        42
Name: count, dtype: int64

PAID: 137
UNPAID: 42
CANCELLED: 56


In [6]:
# Load provider data
print("="*70)
print("LOADING PROVIDER DATA")
print("="*70)

provider_files = glob.glob('td68ff*.csv')
provider_dfs = []
for f in provider_files:
    df = pd.read_csv(f, encoding='utf-8')
    df['source_file'] = f
    provider_dfs.append(df)
    print(f"  {f}: {len(df)} orders")

df_provider = pd.concat(provider_dfs, ignore_index=True)
print(f"\nBefore filtering: {len(df_provider)} orders")

# FILTER: Only dropshipping orders (Dropshpping = 1)
df_provider = df_provider[df_provider['Dropshpping'] == 1]
print(f"After Dropshipping=1 filter: {len(df_provider)} orders")

# REMOVE DUPLICATES: Same order ID appears in multiple files
print(f"Unique Order IDs: {df_provider['Id. Ordine'].nunique()}")
print(f"Duplicate rows: {len(df_provider) - df_provider['Id. Ordine'].nunique()}")
df_provider = df_provider.drop_duplicates(subset=['Id. Ordine'], keep='first')
print(f"After deduplication: {len(df_provider)} unique orders")

df_provider['Data'] = pd.to_datetime(df_provider['Data'])

# Map country names to codes
country_mapping = {
    'Germany': 'DE',
    'Austria': 'AT',
    'Italy': 'IT',
    'Italia': 'IT',
    'France': 'FR',
    'Poland': 'PL',
    'Czech Republic': 'CZ',
    'Slovakia': 'SK',
    'Eslovaquia': 'SK'
}
df_provider['country_code'] = df_provider['Paese'].map(country_mapping)

# Create matching keys
df_provider['full_name'] = (df_provider['Nome'].fillna('') + ' ' + df_provider['Cognome'].fillna('')).str.strip().str.lower()
df_provider['city'] = df_provider['Città'].fillna('').str.strip().str.lower()

print(f"\nTotal provider cost (Base imponibile): €{df_provider['Base imponibile'].sum():.2f}")
print(f"\nProvider orders by country:")
print(df_provider['country_code'].value_counts())

LOADING PROVIDER DATA
  td68ff58fc3bce20.58340114.csv: 100 orders
  td68ff59044314c0.72247156.csv: 200 orders
  td68ff590d08c576.24738580.csv: 246 orders

Before filtering: 546 orders
After Dropshipping=1 filter: 522 orders
Unique Order IDs: 233
Duplicate rows: 289
After deduplication: 233 unique orders

Total provider cost (Base imponibile): €9452.62

Provider orders by country:
country_code
DE    48
IT    46
PL    41
SK    35
CZ    28
AT    27
FR     8
Name: count, dtype: int64


In [7]:
# Match marketplace orders with provider orders
print("="*70)
print("MATCHING MARKETPLACE ORDERS WITH PROVIDER COSTS")
print("="*70)

# Create matching keys for marketplace orders
all_orders['full_name'] = (all_orders['shipping.first_name'].fillna('') + ' ' + all_orders['shipping.last_name'].fillna('')).str.strip().str.lower()
all_orders['city'] = all_orders['shipping.city'].fillna('').str.strip().str.lower()

# Match orders
matched_orders = []
unmatched_orders = []

for idx, order in all_orders.iterrows():
    # Try to find matching provider order
    # Method 1: Match by name + country + date (for GMU orders with customer info)
    if order['full_name'] and order['full_name'] != '':
        potential_matches = df_provider[
            (df_provider['full_name'] == order['full_name']) &
            (df_provider['country_code'] == order['country']) &
            (abs((df_provider['Data'] - order['order_date']).dt.days) <= 2)
        ]
    else:
        # Method 2: Match by country + date + similar amount (for df_10000-only orders without names)
        # Allow 5% tolerance on amount
        order_amount_eur = order['sum_price_gross'] / EXCHANGE_RATES.get(order['country'], 1.0)
        potential_matches = df_provider[
            (df_provider['country_code'] == order['country']) &
            (abs((df_provider['Data'] - order['order_date']).dt.days) <= 2) &
            (abs(df_provider['Totale'] - order_amount_eur) / order_amount_eur <= 0.15)  # 15% tolerance
        ]
    
    if len(potential_matches) > 0:
        # Take the closest match (by date, then by amount if available)
        potential_matches = potential_matches.copy()
        potential_matches['date_diff'] = abs((potential_matches['Data'] - order['order_date']).dt.days)
        
        # If we have amount, also consider amount difference
        if order['full_name'] == '':
            order_amount_eur = order['sum_price_gross'] / EXCHANGE_RATES.get(order['country'], 1.0)
            potential_matches['amount_diff'] = abs(potential_matches['Totale'] - order_amount_eur)
            best_match = potential_matches.sort_values(['date_diff', 'amount_diff']).iloc[0]
        else:
            best_match = potential_matches.sort_values('date_diff').iloc[0]
        
        matched_orders.append({
            **order.to_dict(),
            'provider_order_id': best_match['Id. Ordine'],
            'provider_cost': best_match['Base imponibile'],  # Use Base imponibile as cost
            'provider_date': best_match['Data'],
            'provider_status': best_match["Stato dell'ordine"],
            'provider_totale': best_match['Totale'],
            'provider_shipping': best_match['Spese di spedizione'],
            'match_quality': 'name_country_date' if order['full_name'] != '' else 'amount_country_date'
        })
    else:
        # No match found
        unmatched_orders.append({
            **order.to_dict(),
            'provider_order_id': None,
            'provider_cost': 0,
            'provider_date': None,
            'provider_status': 'NOT_MATCHED',
            'provider_totale': 0,
            'provider_shipping': 0,
            'match_quality': 'no_match'
        })

# Combine matched and unmatched
all_orders_with_costs = pd.DataFrame(matched_orders + unmatched_orders)

print(f"\nMatched orders: {len(matched_orders)}")
print(f"  - By name+country+date: {len([o for o in matched_orders if o['match_quality'] == 'name_country_date'])}")
print(f"  - By amount+country+date: {len([o for o in matched_orders if o['match_quality'] == 'amount_country_date'])}")
print(f"Unmatched orders: {len(unmatched_orders)}")
print(f"Match rate: {len(matched_orders)/len(all_orders)*100:.1f}%")

MATCHING MARKETPLACE ORDERS WITH PROVIDER COSTS

Matched orders: 126
  - By name+country+date: 115
  - By amount+country+date: 11
Unmatched orders: 109
Match rate: 53.6%


In [8]:
# Apply currency conversion
print("="*70)
print("APPLYING CURRENCY CONVERSION")
print("="*70)

# Convert to EUR based on country
all_orders_with_costs['exchange_rate'] = all_orders_with_costs['country'].map(EXCHANGE_RATES)

# Convert marketplace revenue to EUR
all_orders_with_costs['payout_eur'] = all_orders_with_costs['payout'] / all_orders_with_costs['exchange_rate']
all_orders_with_costs['price_gross_eur'] = all_orders_with_costs['price_gross'] / all_orders_with_costs['exchange_rate']
all_orders_with_costs['sum_price_gross_eur'] = all_orders_with_costs['sum_price_gross'] / all_orders_with_costs['exchange_rate']
all_orders_with_costs['fee_gross_eur'] = all_orders_with_costs['fee_gross'] / all_orders_with_costs['exchange_rate']

# Provider costs are already in EUR
all_orders_with_costs['provider_cost_eur'] = all_orders_with_costs['provider_cost']

# Calculate profit
all_orders_with_costs['profit_eur'] = all_orders_with_costs['payout_eur'] - all_orders_with_costs['provider_cost_eur']
all_orders_with_costs['profit_margin_%'] = (all_orders_with_costs['profit_eur'] / all_orders_with_costs['payout_eur'] * 100).fillna(0)

print("\nCurrency conversion applied:")
for country, rate in EXCHANGE_RATES.items():
    orders_count = len(all_orders_with_costs[all_orders_with_costs['country'] == country])
    if orders_count > 0:
        print(f"  {country}: {orders_count} orders, rate={rate}")

APPLYING CURRENCY CONVERSION

Currency conversion applied:
  PL: 59 orders, rate=4.2
  CZ: 39 orders, rate=24.3
  DE: 42 orders, rate=1.0
  FR: 11 orders, rate=1.0
  IT: 7 orders, rate=1.0
  AT: 32 orders, rate=1.0
  SK: 45 orders, rate=1.0


In [9]:
# Calculate profit metrics with proper status split
print("="*70)
print("PROFIT ANALYSIS SUMMARY")
print("="*70)

# Split by order status
paid_orders = all_orders_with_costs[all_orders_with_costs['order_status'] == 'PAID'].copy()
unpaid_orders = all_orders_with_costs[all_orders_with_costs['order_status'] == 'UNPAID'].copy()
cancelled_orders = all_orders_with_costs[all_orders_with_costs['order_status'] == 'CANCELLED'].copy()

# Overall metrics (PAID + UNPAID, excluding CANCELLED)
active_orders = all_orders_with_costs[all_orders_with_costs['order_status'].isin(['PAID', 'UNPAID'])].copy()
total_revenue_eur = active_orders['payout_eur'].sum()
total_costs_eur = active_orders['provider_cost_eur'].sum()
total_profit_eur = active_orders['profit_eur'].sum()

# PAID metrics
paid_revenue_eur = paid_orders['payout_eur'].sum()
paid_costs_eur = paid_orders['provider_cost_eur'].sum()
paid_profit_eur = paid_orders['profit_eur'].sum()

# UNPAID metrics
unpaid_revenue_eur = unpaid_orders['payout_eur'].sum()
unpaid_costs_eur = unpaid_orders['provider_cost_eur'].sum()
unpaid_profit_eur = unpaid_orders['profit_eur'].sum()

# CANCELLED metrics
cancelled_revenue_eur = cancelled_orders['payout_eur'].sum()
cancelled_costs_eur = cancelled_orders['provider_cost_eur'].sum()
cancelled_profit_eur = cancelled_orders['profit_eur'].sum()

print(f"\n{'='*50}")
print(f"TOTAL (PAID + UNPAID, excluding CANCELLED)")
print(f"{'='*50}")
print(f"Revenue (marketplace payout): €{total_revenue_eur:,.2f}")
print(f"Costs (provider):             €{total_costs_eur:,.2f}")
print(f"Profit:                       €{total_profit_eur:,.2f}")
print(f"Profit margin:                {(total_profit_eur/total_revenue_eur*100) if total_revenue_eur > 0 else 0:.1f}%")
print(f"Orders:                       {len(active_orders)}")

print(f"\n{'='*50}")
print(f"PAID ORDERS (Already Received Payment)")
print(f"{'='*50}")
print(f"Revenue (marketplace payout): €{paid_revenue_eur:,.2f}")
print(f"Costs (provider):             €{paid_costs_eur:,.2f}")
print(f"REALIZED PROFIT:              €{paid_profit_eur:,.2f}")
print(f"Profit margin:                {(paid_profit_eur/paid_revenue_eur*100) if paid_revenue_eur > 0 else 0:.1f}%")
print(f"Orders:                       {len(paid_orders)}")

print(f"\n{'='*50}")
print(f"UNPAID ORDERS (Pending Payment)")
print(f"{'='*50}")
print(f"Expected revenue:             €{unpaid_revenue_eur:,.2f}")
print(f"Costs (provider):             €{unpaid_costs_eur:,.2f}")
print(f"OUTSTANDING PROFIT:           €{unpaid_profit_eur:,.2f}")
print(f"Profit margin:                {(unpaid_profit_eur/unpaid_revenue_eur*100) if unpaid_revenue_eur > 0 else 0:.1f}%")
print(f"Orders:                       {len(unpaid_orders)}")

print(f"\n{'='*50}")
print(f"CANCELLED ORDERS")
print(f"{'='*50}")
print(f"Lost revenue:                 €{cancelled_revenue_eur:,.2f}")
print(f"Costs (provider):             €{cancelled_costs_eur:,.2f}")
print(f"Loss:                         €{cancelled_profit_eur:,.2f}")
print(f"Orders:                       {len(cancelled_orders)}")

print(f"\n{'='*50}")
print(f"EXPECTED TOTAL (if all unpaid orders are paid)")
print(f"{'='*50}")
expected_total_profit = paid_profit_eur + unpaid_profit_eur
print(f"Expected total profit:        €{expected_total_profit:,.2f}")

PROFIT ANALYSIS SUMMARY

TOTAL (PAID + UNPAID, excluding CANCELLED)
Revenue (marketplace payout): €8,214.23
Costs (provider):             €4,101.50
Profit:                       €4,112.73
Profit margin:                50.1%
Orders:                       179

PAID ORDERS (Already Received Payment)
Revenue (marketplace payout): €6,156.76
Costs (provider):             €3,934.65
REALIZED PROFIT:              €2,222.11
Profit margin:                36.1%
Orders:                       137

UNPAID ORDERS (Pending Payment)
Expected revenue:             €2,057.47
Costs (provider):             €166.85
OUTSTANDING PROFIT:           €1,890.62
Profit margin:                91.9%
Orders:                       42

CANCELLED ORDERS
Lost revenue:                 €5,326.68
Costs (provider):             €441.60
Loss:                         €4,885.08
Orders:                       56

EXPECTED TOTAL (if all unpaid orders are paid)
Expected total profit:        €4,112.73


In [10]:
# Monthly profit analysis by country
print("="*70)
print("MONTHLY PROFIT BREAKDOWN BY COUNTRY")
print("="*70)

monthly_profit = all_orders_with_costs.pivot_table(
    values='profit_eur',
    index='month',
    columns='country',
    aggfunc='sum',
    fill_value=0
).round(2)

monthly_profit['TOTAL'] = monthly_profit.sum(axis=1)

print("\nMonthly Profit by Country (EUR):")
print(monthly_profit)

# Monthly profit by payment status
monthly_profit_by_status = all_orders_with_costs.pivot_table(
    values='profit_eur',
    index='month',
    columns='is_paid',
    aggfunc='sum',
    fill_value=0
).round(2)

monthly_profit_by_status.columns = ['UNPAID (Outstanding)', 'PAID (Realized)']
monthly_profit_by_status['TOTAL'] = monthly_profit_by_status.sum(axis=1)

print("\nMonthly Profit by Payment Status (EUR):")
print(monthly_profit_by_status)

MONTHLY PROFIT BREAKDOWN BY COUNTRY

Monthly Profit by Country (EUR):
country      AT      CZ      DE      FR      IT      PL       SK    TOTAL
month                                                                    
2025-07   97.94  302.28    0.00    0.00    0.00  717.02  1587.36  2704.60
2025-08   81.36    3.17   75.50  205.34    0.00   61.01   255.13   681.51
2025-09  383.21  396.73  103.19   56.63  -15.44  963.74   421.18  2309.24
2025-10  551.53  607.36  503.76  282.67  244.55  882.08   230.52  3302.47

Monthly Profit by Payment Status (EUR):
         UNPAID (Outstanding)  PAID (Realized)    TOTAL
month                                                  
2025-07               2684.25            20.35  2704.60
2025-08                497.33           184.18   681.51
2025-09               1335.07           974.17  2309.24
2025-10               2259.06          1043.42  3302.48


In [11]:
# Country-level summary
print("="*70)
print("PROFIT BY COUNTRY")
print("="*70)

country_summary = all_orders_with_costs.groupby('country').agg({
    'order_number': 'count',
    'payout_eur': 'sum',
    'provider_cost_eur': 'sum',
    'profit_eur': 'sum',
    'is_paid': 'sum'
}).round(2)

country_summary.columns = ['Orders', 'Revenue (EUR)', 'Costs (EUR)', 'Profit (EUR)', 'Paid Orders']
country_summary['Unpaid Orders'] = country_summary['Orders'] - country_summary['Paid Orders']
country_summary['Profit Margin %'] = (country_summary['Profit (EUR)'] / country_summary['Revenue (EUR)'] * 100).round(1)

print("\n", country_summary.sort_values('Profit (EUR)', ascending=False))

PROFIT BY COUNTRY

          Orders  Revenue (EUR)  Costs (EUR)  Profit (EUR)  Paid Orders  \
country                                                                  
PL           59        3997.25      1373.40       2623.85           32   
SK           45        3492.17       997.98       2494.19           31   
CZ           39        1896.93       587.40       1309.53           19   
AT           32        1864.12       750.08       1114.04           24   
DE           42        1325.43       642.98        682.45           29   
FR           11         706.55       161.91        544.64            1   
IT            7         258.46        29.35        229.11            1   

         Unpaid Orders  Profit Margin %  
country                                  
PL                  27             65.6  
SK                  14             71.4  
CZ                  20             69.0  
AT                   8             59.8  
DE                  13             51.5  
FR                 

In [12]:
# Prepare detailed lists for each status
print("="*70)
print("PREPARING DETAILED ORDER LISTS")
print("="*70)

# UNPAID orders detailed list
unpaid_detailed = unpaid_orders[[
    'order_number', 'country', 'booking_date', 'order_date',
    'shipping.first_name', 'shipping.last_name', 'shipping.city',
    'title_item', 'price_gross', 'sum_price_gross', 'payout',
    'price_gross_eur', 'sum_price_gross_eur', 'payout_eur',
    'provider_cost_eur', 'profit_eur', 'profit_margin_%',
    'provider_order_id', 'provider_status', 'match_quality', 'source'
]].copy()
unpaid_detailed['days_since_order'] = (pd.Timestamp.now() - unpaid_detailed['booking_date']).dt.days
unpaid_detailed = unpaid_detailed.sort_values(['country', 'booking_date'], ascending=[True, False])

print(f"\nUNPAID ORDERS: {len(unpaid_detailed)}")
print(f"Outstanding revenue: €{unpaid_detailed['payout_eur'].sum():,.2f}")
print(f"Outstanding costs: €{unpaid_detailed['provider_cost_eur'].sum():,.2f}")
print(f"Outstanding profit: €{unpaid_detailed['profit_eur'].sum():,.2f}")
print(f"\nBy country:")
print(unpaid_detailed.groupby('country').agg({
    'order_number': 'count',
    'payout_eur': 'sum',
    'provider_cost_eur': 'sum',
    'profit_eur': 'sum'
}).round(2))

# CANCELLED orders detailed list
cancelled_detailed = cancelled_orders[[
    'order_number', 'country', 'booking_date', 'order_date',
    'shipping.first_name', 'shipping.last_name', 'shipping.city',
    'title_item', 'price_gross', 'sum_price_gross', 'payout',
    'price_gross_eur', 'sum_price_gross_eur', 'payout_eur',
    'provider_cost_eur', 'profit_eur',
    'provider_order_id', 'provider_status', 'match_quality', 'source'
]].copy()
cancelled_detailed['days_since_order'] = (pd.Timestamp.now() - cancelled_detailed['booking_date']).dt.days
cancelled_detailed = cancelled_detailed.sort_values(['country', 'booking_date'], ascending=[True, False])

print(f"\nCANCELLED ORDERS: {len(cancelled_detailed)}")
print(f"Lost revenue: €{cancelled_detailed['payout_eur'].sum():,.2f}")
print(f"Costs incurred: €{cancelled_detailed['provider_cost_eur'].sum():,.2f}")
print(f"Net loss: €{cancelled_detailed['profit_eur'].sum():,.2f}")
print(f"\nBy country:")
print(cancelled_detailed.groupby('country').agg({
    'order_number': 'count',
    'payout_eur': 'sum',
    'provider_cost_eur': 'sum',
    'profit_eur': 'sum'
}).round(2))

PREPARING DETAILED ORDER LISTS

UNPAID ORDERS: 42
Outstanding revenue: €2,057.47
Outstanding costs: €166.85
Outstanding profit: €1,890.62

By country:
         order_number  payout_eur  provider_cost_eur  profit_eur
country                                                         
AT                  3      227.26               0.00      227.26
CZ                  9      559.89              51.05      508.84
DE                 10      388.65               0.00      388.65
FR                  7      343.17             115.80      227.37
IT                  3       81.76               0.00       81.76
PL                  7      355.79               0.00      355.79
SK                  3      100.95               0.00      100.95

CANCELLED ORDERS: 56
Lost revenue: €5,326.68
Costs incurred: €441.60
Net loss: €4,885.08

By country:
         order_number  payout_eur  provider_cost_eur  profit_eur
country                                                         
AT                  5      275.

In [13]:
# Export to Excel - CORRECTED VERSION
print("="*70)
print("EXPORTING TO EXCEL")
print("="*70)

excel_filename = 'profit_analysis_all_countries.xlsx'
writer = pd.ExcelWriter(excel_filename, engine='xlsxwriter')

# Tab 1: Executive Summary
print("\n1. Executive Summary...")
summary_data = pd.DataFrame({
    'Metric': [
        'Total Active Orders (PAID + UNPAID)',
        'PAID Orders',
        'UNPAID Orders',
        'CANCELLED Orders',
        '',
        'ACTIVE - Total Revenue (EUR)',
        'ACTIVE - Total Costs (EUR)',
        'ACTIVE - Total Profit (EUR)',
        'ACTIVE - Profit Margin %',
        '',
        'PAID - Revenue (EUR)',
        'PAID - Costs (EUR)',
        'PAID - REALIZED PROFIT (EUR)',
        'PAID - Profit Margin %',
        '',
        'UNPAID - Expected Revenue (EUR)',
        'UNPAID - Costs (EUR)',
        'UNPAID - OUTSTANDING PROFIT (EUR)',
        'UNPAID - Profit Margin %',
        '',
        'CANCELLED - Lost Revenue (EUR)',
        'CANCELLED - Costs Incurred (EUR)',
        'CANCELLED - Net Loss (EUR)',
        '',
        'EXPECTED TOTAL PROFIT (EUR)',
        '(if all unpaid orders are paid)'
    ],
    'Value': [
        len(active_orders),
        len(paid_orders),
        len(unpaid_orders),
        len(cancelled_orders),
        '',
        f"{total_revenue_eur:,.2f}",
        f"{total_costs_eur:,.2f}",
        f"{total_profit_eur:,.2f}",
        f"{(total_profit_eur/total_revenue_eur*100):.1f}%" if total_revenue_eur > 0 else '0%',
        '',
        f"{paid_revenue_eur:,.2f}",
        f"{paid_costs_eur:,.2f}",
        f"{paid_profit_eur:,.2f}",
        f"{(paid_profit_eur/paid_revenue_eur*100):.1f}%" if paid_revenue_eur > 0 else '0%',
        '',
        f"{unpaid_revenue_eur:,.2f}",
        f"{unpaid_costs_eur:,.2f}",
        f"{unpaid_profit_eur:,.2f}",
        f"{(unpaid_profit_eur/unpaid_revenue_eur*100):.1f}%" if unpaid_revenue_eur > 0 else '0%',
        '',
        f"{cancelled_revenue_eur:,.2f}",
        f"{cancelled_costs_eur:,.2f}",
        f"{cancelled_profit_eur:,.2f}",
        '',
        f"{expected_total_profit:,.2f}",
        ''
    ]
})
summary_data.to_excel(writer, sheet_name='1_Executive_Summary', index=False)

# Tab 2: Profit by Country
print("2. Profit by Country...")
country_summary.to_excel(writer, sheet_name='2_Profit_by_Country')

# Tab 3: Monthly Profit by Country
print("3. Monthly Profit by Country...")
monthly_profit_export = monthly_profit.copy()
monthly_profit_export.index = monthly_profit_export.index.astype(str)
monthly_profit_export.to_excel(writer, sheet_name='3_Monthly_Profit_Country')

# Tab 4: Monthly Profit by Status
print("4. Monthly Profit by Status...")
monthly_status_export = monthly_profit_by_status.copy()
monthly_status_export.index = monthly_status_export.index.astype(str)
monthly_status_export.to_excel(writer, sheet_name='4_Monthly_Profit_Status')

# Tab 5: UNPAID Orders - Full Details
print("5. UNPAID Orders - Full Details...")
unpaid_detailed.to_excel(writer, sheet_name='5_UNPAID_Orders_Details', index=False)

# Tab 6: CANCELLED Orders - Full Details
print("6. CANCELLED Orders - Full Details...")
if len(cancelled_detailed) > 0:
    cancelled_detailed.to_excel(writer, sheet_name='6_CANCELLED_Orders', index=False)
else:
    pd.DataFrame({'Message': ['No cancelled orders']}).to_excel(writer, sheet_name='6_CANCELLED_Orders', index=False)

# Tab 7: PAID Orders
print("7. PAID Orders...")
paid_detailed = paid_orders[[
    'order_number', 'country', 'booking_date', 'order_date',
    'title_item', 'payout', 'payout_eur', 'provider_cost_eur',
    'profit_eur', 'profit_margin_%', 'provider_order_id', 'match_quality'
]].copy().sort_values(['country', 'booking_date'], ascending=[True, False])
paid_detailed.to_excel(writer, sheet_name='7_PAID_Orders', index=False)

# Tab 8: All Orders with Profit
print("8. All Orders with Profit...")
all_orders_export = all_orders_with_costs[[
    'order_number', 'country', 'order_status', 'booking_date', 'order_date',
    'title_item', 'payout', 'payout_eur', 'provider_cost_eur',
    'profit_eur', 'profit_margin_%', 'provider_order_id',
    'provider_status', 'match_quality', 'source'
]].copy()
all_orders_export = all_orders_export.sort_values(['order_status', 'country', 'booking_date'], ascending=[True, True, False])
all_orders_export.to_excel(writer, sheet_name='8_All_Orders_Profit', index=False)

# Tab 9: Unmatched Orders
print("9. Unmatched Orders...")
unmatched_export = all_orders_with_costs[all_orders_with_costs['match_quality'] == 'no_match'][[
    'order_number', 'country', 'order_status', 'booking_date', 'full_name', 'city',
    'payout_eur', 'source'
]].copy()
unmatched_export.to_excel(writer, sheet_name='9_Unmatched_Orders', index=False)

# Tab 10: Provider Orders
print("10. Provider Orders...")
provider_export = df_provider[[
    'Id. Ordine', 'Data', 'Nome', 'Cognome', 'Paese', 'country_code',
    'Totale', 'Base imponibile', 'Spese di spedizione',
    "Stato dell'ordine", 'Fattura'
]].copy().sort_values('Data', ascending=False)
provider_export.to_excel(writer, sheet_name='10_Provider_Orders', index=False)

writer.close()

print(f"\n✓ Excel file created: {excel_filename}")
print("\n" + "="*70)
print("ANALYSIS COMPLETE!")
print("="*70)
print(f"\nKey Findings:")
print(f"  • Active Orders: {len(active_orders)} (PAID: {len(paid_orders)}, UNPAID: {len(unpaid_orders)})")
print(f"  • Cancelled Orders: {len(cancelled_orders)}")
print(f"  • Realized Profit (PAID): €{paid_profit_eur:,.2f} (Margin: {(paid_profit_eur/paid_revenue_eur*100) if paid_revenue_eur > 0 else 0:.1f}%)")
print(f"  • Outstanding Profit (UNPAID): €{unpaid_profit_eur:,.2f} (Margin: {(unpaid_profit_eur/unpaid_revenue_eur*100) if unpaid_revenue_eur > 0 else 0:.1f}%)")
print(f"  • Expected Total Profit: €{expected_total_profit:,.2f}")
print(f"  • Cancelled Loss: €{cancelled_profit_eur:,.2f}")

EXPORTING TO EXCEL

1. Executive Summary...
2. Profit by Country...
3. Monthly Profit by Country...
4. Monthly Profit by Status...
5. UNPAID Orders - Full Details...
6. CANCELLED Orders - Full Details...
7. PAID Orders...
8. All Orders with Profit...
9. Unmatched Orders...
10. Provider Orders...

✓ Excel file created: profit_analysis_all_countries.xlsx

ANALYSIS COMPLETE!

Key Findings:
  • Active Orders: 179 (PAID: 137, UNPAID: 42)
  • Cancelled Orders: 56
  • Realized Profit (PAID): €2,222.11 (Margin: 36.1%)
  • Outstanding Profit (UNPAID): €1,890.62 (Margin: 91.9%)
  • Expected Total Profit: €4,112.73
  • Cancelled Loss: €4,885.08
